# Flagship Backup #1 (C2): Witness-Bottlenecked Self-Verification Kill Experiment
**Paper:** Witness-Bottlenecked GRPO (`WB-GRPO`): Anti-Steganography and Robust Reasoning Verification Under Derivation Erasure
**Target Venue:** NeurIPS 2027 Flagship Submission
**Lead Researcher:** Jaime Jarauta

### One-Click Execution
Press **Runtime -> Run all** (`Cmd+F9` / `Ctrl+F9`).
This notebook self-heals, downloads `Qwen/Qwen2.5-Math-1.5B-Instruct`, evaluates across MATH-500 with exact conditional independence (derivation erasure $D_1 = \emptyset$ and witness truncation $W_1 \le 64$ tokens), reports stratified Macro-F1 across invertible vs. non-invertible math problems, and automatically unassigns the GPU when finished.

In [ ]:
# Step 1: Self-Healing Environment Setup & Credit Guard
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print('\n[Credit Guard] Unhandled error or interrupt! Unassigning Colab GPU...')
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), lambda self, et, val, tb, tb_offset=None: (self.showtraceback((et, val, tb), tb_offset=tb_offset), auto_teardown()))

print('[Ready] Credit guard active. VM will self-terminate on error or completion.')


In [ ]:
# Step 2: Clone Repository & Install Dependencies
import os, sys, subprocess

repo_url = 'https://github.com/jaimejarauta/decoupled-delta-net.git'
branch = 'feature/wb-witness-verify'
target_dir = '/content/wb-witness-verify'

%cd /content
if not os.path.exists(target_dir):
    print(f'Cloning wb-witness-verify from {repo_url} (branch: {branch})...')
    subprocess.run(['git', 'clone', '-b', branch, repo_url, target_dir], check=True)
else:
    print(f'Updating wb-witness-verify...')
    subprocess.run(['git', '-C', target_dir, 'fetch', 'origin', branch], check=True)
    subprocess.run(['git', '-C', target_dir, 'checkout', branch], check=True)
    subprocess.run(['git', '-C', target_dir, 'pull', 'origin', branch], check=True)

%cd /content/wb-witness-verify
!pip install -q -e .
!pip install -q "transformers>=4.48" "datasets>=3.0" accelerate pytest

sys.path.insert(0, '/content/wb-witness-verify/src')
os.environ['PYTHONPATH'] = f'/content/wb-witness-verify/src:{os.environ.get("PYTHONPATH", "")}'
print('[Ready] wb-witness-verify ready on GPU.')


In [ ]:
# Step 3: Run Fast CPU Pre-Flight Checks
!pytest -q tests/test_wb_verify.py


In [ ]:
# Step 4: Execute Flagship Option C2 Kill Experiment (MATH-500 Witness Bottleneck)
# Evaluates reasoning problems across:
# 1. prompted_2call (Clean witness-only verifier: D1 erased, W1 <= 64 toks)
# 2. answer_only (Ablation: zero witness budget, W1 = empty)
# 3. full_trace_prompted (Polluted baseline: D1 preserved, subject to confirmation bias / steganography)
# 4. majority2 (Standard test-time compute baseline: Majority@2 voting)

import os
os.makedirs('results', exist_ok=True)
!python scripts/run_kill_c2.py eval \
    --model Qwen/Qwen2.5-Math-1.5B-Instruct \
    --n 50 \
    --out results/kill_c2.jsonl


In [ ]:
# Step 5: Generate Pre-Registered Verdict & Stratified Invertibility Report
import os
if os.path.exists('results/kill_c2.jsonl'):
    !python scripts/run_kill_c2.py report --out results/kill_c2.jsonl
else:
    print('[Warning] results/kill_c2.jsonl not found. Check Step 4 logs.')


In [ ]:
# Step 6: Credit Guard - Automatically Unassign GPU to Save Credits
print('\n[Credit Guard] Flagship experiment finished! Unassigning Colab GPU immediately...')
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f'Teardown notice: {e}')
